### 1.环境检查

In [2]:
from pathlib import Path
import sys
import platform
import importlib.metadata as metadata

PROJECT = Path("/home/qiuyuhang_26/workspace/VLM-test")
DATASET_ROOT = PROJECT / "dataset"
RESULTS_ROOT = PROJECT / "results"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Project exists:", PROJECT.exists())
print("Dataset exists:", DATASET_ROOT.exists())

print("\n=== Package versions ===")

packages = [
    "torch",
    "transformers",
    "accelerate",
    "datasets",
    "Pillow",
    "ipykernel",
    "jupyter",
    "lmms-eval",
]

for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\n=== GPU information ===")

import torch

print("PyTorch:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

    free, total = torch.cuda.mem_get_info()
    print(f"GPU free memory: {free / 1024**3:.2f} GiB")
    print(f"GPU total memory: {total / 1024**3:.2f} GiB")

Python executable: /home/qiuyuhang_26/opt/miniconda3/envs/vlm-test/bin/python
Python version: 3.11.17 (main, Oct  1 2026, 20:17:42) [GCC 14.3.0]
Platform: Linux-6.2.0-37-generic-x86_64-with-glibc2.35
Project exists: True
Dataset exists: True

=== Package versions ===
torch: 2.10.0+cu128
transformers: 4.57.6
accelerate: 1.15.0
datasets: 4.8.5
Pillow: 12.3.0
ipykernel: 7.3.0
jupyter: NOT INSTALLED
lmms-eval: NOT INSTALLED

=== GPU information ===
PyTorch: 2.10.0+cu128
PyTorch CUDA runtime: 12.8
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
BF16 supported: True
GPU free memory: 22.26 GiB
GPU total memory: 23.52 GiB


### 2.检查数据集

In [3]:
from collections import Counter, defaultdict

for dataset_name in ["CV-Bench", "SPAR-Bench", "VSI-Bench"]:
    root = DATASET_ROOT / dataset_name

    print(f"\n{'=' * 60}")
    print(f"DATASET: {dataset_name}")
    print("Path:", root)
    print("Exists:", root.exists())

    if not root.exists():
        continue

    print("\nTop-level files:")
    for path in sorted(root.iterdir())[:30]:
        kind = "DIR " if path.is_dir() else "FILE"
        print(f"  [{kind}] {path.name}")

    extensions = Counter()
    examples = defaultdict(list)

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        ext = path.suffix.lower() or "[no extension]"
        extensions[ext] += 1

        important = {
            ".json", ".jsonl", ".csv", ".parquet",
            ".yaml", ".yml", ".mp4", ".avi",
            ".jpg", ".jpeg", ".png", ".webp",
            ".npy", ".npz",
        }

        if ext in important and len(examples[ext]) < 3:
            examples[ext].append(
                str(path.relative_to(root))
            )

    print("\nFile counts by extension:")
    print(extensions.most_common(20))

    print("\nExample files:")
    for ext, files in sorted(examples.items()):
        for filename in files:
            print(f"  {ext}: {filename}")


DATASET: CV-Bench
Path: /home/qiuyuhang_26/workspace/VLM-test/dataset/CV-Bench
Exists: True

Top-level files:
  [DIR ] .cache
  [FILE] .gitattributes
  [FILE] README.md
  [FILE] build_img.py
  [DIR ] samples
  [FILE] test_2d.jsonl
  [FILE] test_2d.parquet
  [FILE] test_3d.jsonl
  [FILE] test_3d.parquet

File counts by extension:
[('.png', 60), ('.metadata', 7), ('.lock', 7), ('.jsonl', 2), ('.parquet', 2), ('[no extension]', 2), ('.jpg', 2), ('.csv', 2), ('.md', 1), ('.py', 1)]

Example files:
  .csv: samples/3D/annotations.csv
  .csv: samples/2D/annotations.csv
  .jpg: samples/2d_overview.jpg
  .jpg: samples/3d_overview.jpg
  .jsonl: test_3d.jsonl
  .jsonl: test_2d.jsonl
  .parquet: test_3d.parquet
  .parquet: test_2d.parquet
  .png: samples/3D/3d_20.png
  .png: samples/3D/3d_12.png
  .png: samples/3D/3d_05.png

DATASET: SPAR-Bench
Path: /home/qiuyuhang_26/workspace/VLM-test/dataset/SPAR-Bench
Exists: True

Top-level files:
  [DIR ] .cache
  [FILE] .gitattributes
  [FILE] README.md
 